# 08 — A complete forecasting workflow

This capstone notebook chains the whole pipeline on Sioux Falls, mirroring the
AequilibraE *Forecasting* documentation example:

1. **base-year assignment** with skimming;
2. **gravity model calibration** on congested times;
3. **future demand**: grown trip ends balanced with IPF;
4. **future-year assignment** with **select link analysis**;
5. compare base vs future flows on a map.


In [1]:
from pathlib import Path
from tempfile import gettempdir
from uuid import uuid4

import numpy as np
import pandas as pd

from aequilibrae.utils.create_example import create_example
from aequilibrae.paths import TrafficAssignment, TrafficClass

np.random.seed(0)

fldr = str(Path(gettempdir()) / uuid4().hex)
project = create_example(fldr, "sioux_falls")

project.network.build_graphs()
graph = project.network.graphs["c"]
graph.set_graph("free_flow_time")
graph.set_skimming(["free_flow_time", "distance"])
graph.set_blocked_centroid_flows(False)

C:\Users\Riz\Desktop\AequilibraE\aequilibrae\aequilibrae\paths\graph.py:218: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

Try using '.loc[row_indexer, col_indexer] = value' instead, to perform the assignment in a single step.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html#chained-assignment
  build_compressed_graph(self, remove_dead_ends)
C:\Users\Riz\Desktop\AequilibraE\aequilibrae\aequilibrae\paths\graph.py:218: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are settin

In [2]:
# ---- 1. Base year assignment --------------------------------------------
demand = project.matrices.get_matrix("demand_omx")
demand.computational_view(["matrix"])

assigclass = TrafficClass(name="car", graph=graph, matrix=demand)
assig = TrafficAssignment()
assig.add_class(assigclass)
assig.set_vdf("BPR")
assig.set_vdf_parameters({"alpha": "b", "beta": "power"})
assig.set_capacity_field("capacity")
assig.set_time_field("free_flow_time")
assig.set_algorithm("bfw")
assig.max_iter = 500
assig.rgap_target = 0.001
assig.execute()

assig.save_results("base_year_assignment")
assig.save_skims("base_year_skims", which_ones="all", format="omx")
base_flows = assig.results()[["matrix_tot"]].rename(columns={"matrix_tot": "base"})

car                                               :   0%|          | 0/24 [00:00<?, ?it/s]

Equilibrium Assignment                            :   0%|          | 0/500 [00:00<?, ?it/s]

In [3]:
# ---- 2. Gravity calibration on congested skims --------------------------
from aequilibrae.distribution import GravityCalibration

imped = project.matrices.get_matrix("base_year_skims_car")
imped.computational_view(["free_flow_time_final"])   # congested time, last iteration

np.fill_diagonal(imped.matrix_view, 0)
intrazonal = 0.75 * np.amin(imped.matrix_view, where=imped.matrix_view > 0,
                            initial=imped.matrix_view.max(), axis=1)
np.fill_diagonal(imped.matrix_view, intrazonal)
imped.save(names=["time_with_intrazonals"])

gc = GravityCalibration(matrix=demand, impedance=imped, function="power", nan_as_zero=True)
gc.calibrate()
gc.model.save(str(Path(fldr) / "power_model.mod"))

In [4]:
# ---- 3. Future demand: grown vectors + IPF -------------------------------
from aequilibrae.distribution import Ipf

origins = np.sum(demand.matrix_view, axis=1)
destinations = np.sum(demand.matrix_view, axis=0)
orig = origins * (1 + np.random.rand(origins.shape[0]) / 10)
dest = destinations * (1 + np.random.rand(origins.shape[0]) / 10)
dest *= orig.sum() / dest.sum()

vectors = pd.DataFrame({"origins": orig, "destinations": dest}, index=demand.index[:])

ipf = Ipf(matrix=demand, vectors=vectors, row_field="origins",
          column_field="destinations", nan_as_zero=True)
ipf.fit()
ipf.save_to_project(name="demand_future", file_name="demand_future.omx")

In [5]:
# ---- 4. Future-year assignment with select link analysis -----------------
future = project.matrices.get_matrix("demand_future")
future.computational_view("matrix")

assigclass = TrafficClass(name="car", graph=graph, matrix=future)
assigclass.set_select_links({
    "downtown_bridge": [(13, 1), (25, 1)],   # (link_id, direction) tuples
})

assig = TrafficAssignment()
assig.add_class(assigclass)
assig.set_vdf("BPR")
assig.set_vdf_parameters({"alpha": "b", "beta": "power"})
assig.set_capacity_field("capacity")
assig.set_time_field("free_flow_time")
assig.set_algorithm("bfw")
assig.max_iter = 500
assig.rgap_target = 0.001
assig.execute()

assig.save_results("future_year_assignment")
assig.save_select_link_results("select_link_analysis")

future_flows = assig.results()[["matrix_tot"]].rename(columns={"matrix_tot": "future"})

car                                               :   0%|          | 0/24 [00:00<?, ?it/s]

Equilibrium Assignment                            :   0%|          | 0/500 [00:00<?, ?it/s]

In [6]:
# ---- 5. Compare ----------------------------------------------------------
comparison = base_flows.join(future_flows)
comparison["growth_pct"] = (100 * (comparison.future / comparison.base - 1)).round(1)
comparison.sort_values("growth_pct", ascending=False).head(8)

,base,future,growth_pct
link_id,,,
1,4567.230095,5452.502999,19.4
3,4574.375228,5459.857040,19.4
55,15713.659996,18000.222079,14.6
2,8174.162682,9348.015078,14.4
60,19162.405854,21596.144990,12.7
7,10207.585261,11349.011180,11.2
35,10172.571513,11286.688037,11.0
6,14184.787941,15708.398419,10.7


In [7]:
# JupyterGIS map helper ------------------------------------------------------
# GISDocument is JupyterGIS' notebook API: it builds a live, QGIS-like map
# document rendered directly in JupyterLab. Layers added from GeoDataFrames
# are converted to GeoJSON on the fly.
#
# add_gdf also translates the declarative symbology into the OpenLayers
# flat-style expressions the current JupyterGIS frontend renders from, so
# colours and line widths show up without touching the symbology panel.
import json

import matplotlib.colors
import matplotlib.pyplot as _plt
from jupytergis import GISDocument
from jupytergis_lab.notebook.symbology import to_symbology_state

OSM_TILES = "https://tile.openstreetmap.org/{z}/{x}/{y}.png"

def new_map(gdf_for_extent=None, zoom=12):
    """Create a GISDocument centred on a layer, with an OpenStreetMap basemap."""
    kwargs = {}
    if gdf_for_extent is not None:
        b = gdf_for_extent.total_bounds  # (minx, miny, maxx, maxy)
        kwargs = {"longitude": (b[0] + b[2]) / 2, "latitude": (b[1] + b[3]) / 2, "zoom": zoom}
    doc = GISDocument(**kwargs)
    doc.add_raster_layer(OSM_TILES, name="OpenStreetMap", attribution="(C) OpenStreetMap contributors", opacity=0.6)
    return doc

def _hex(rgba):
    return matplotlib.colors.to_hex(rgba)

def _ramp_expr(fld, params):
    name, dom = params.get("name", "viridis"), params.get("domain") or [0.0, 1.0]
    cmap = _plt.get_cmap(name)
    if params.get("reverse"):
        cmap = cmap.reversed()
    expr = ["interpolate", ["linear"], ["get", fld]]
    for i in range(7):
        t = i / 6
        expr += [dom[0] + t * (dom[1] - dom[0]), _hex(cmap(t))]
    return expr

def _scalar_expr(fld, params):
    d, r = params["domain"], params["range"]
    return ["interpolate", ["linear"], ["get", fld], d[0], r[0], d[1], r[1]]

def _cat_expr(fld, params, gdf):
    cmap = _plt.get_cmap(params.get("colorRamp", "tab10"))
    vals = list(dict.fromkeys(gdf[fld].dropna()))
    expr = ["match", ["get", fld]]
    for i, v in enumerate(vals):
        expr += [v, _hex(cmap(i % cmap.N))]
    return expr + ["#9ca3af"]

def _flat_style(symbology, gdf):
    """Grammar symbology -> OpenLayers flat-style dict (what the map renders)."""
    state = to_symbology_state(symbology)
    if not state:
        return None
    flat = {}
    for layer in state.get("layers", []):
        for rule in layer.get("rules", []):
            flds = rule.get("fields") or [None]
            for m in rule.get("mappings", []):
                scheme = m["scale"]["scheme"]
                params = m["scale"].get("params", {})
                if scheme == "constant_rgba":
                    val = params["value"]
                    val = _hex([c if c <= 1 else c / 255 for c in val]) if isinstance(val, (list, tuple)) else val
                elif scheme == "constant_num":
                    val = params["value"]
                elif scheme == "colorMap":
                    val = _ramp_expr(flds[0], params)
                elif scheme == "scalar":
                    val = _scalar_expr(flds[0], params)
                elif scheme == "categorical":
                    val = _cat_expr(flds[0], params, gdf)
                else:
                    continue
                for enc in m.get("encodings", []):
                    flat[enc] = val
    if any(k.startswith("circle") for k in flat) and "circle-radius" not in flat:
        flat["circle-radius"] = 5
    if "stroke-color" in flat and "stroke-width" not in flat:
        flat["stroke-width"] = 1.5
    return flat or None

def merge_lines(gdf, tol=0.01):
    """Collapse many lines into a single MultiLineString feature.

    Backdrop and class-display layers do not need per-feature identity, and one
    merged feature is a fraction of the size of tens of thousands of features -
    which keeps national-scale layers inside the notebook sync message limit.
    """
    import geopandas as _gpd
    from shapely.geometry import MultiLineString
    parts = []
    for geom in gdf.geometry.simplify(tol):
        if geom is None or geom.is_empty:
            continue
        parts.extend(geom.geoms if geom.geom_type == "MultiLineString" else [geom])
    return _gpd.GeoDataFrame({"links": [len(parts)]}, geometry=[MultiLineString(parts)], crs=gdf.crs)

def _round_coords(o, nd=5):
    if isinstance(o, (int, float)):
        return round(o, nd)
    if isinstance(o, list):
        return [_round_coords(v, nd) for v in o]
    return o

def add_gdf(doc, gdf, name, symbology=None, **kwargs):
    """Add a GeoDataFrame to the map as a GeoJSON layer, with rendered symbology.

    Coordinates are quantized to ~1 m so even national-scale layers stay well
    under Jupyter's websocket message limit (oversized layers are dropped
    silently by the sync, so this matters more than it looks).
    """
    data = json.loads(gdf.to_json())
    for f in data.get("features", []):
        g = f.get("geometry")
        if g and "coordinates" in g:
            g["coordinates"] = _round_coords(g["coordinates"])
    lid = doc.add_geojson_layer(data=data, name=name,
                                symbology=symbology, **kwargs)
    flat = _flat_style(symbology, gdf)
    if flat:
        layer = doc._layers.get(lid)
        layer["parameters"]["color"] = flat
        doc._layers[lid] = layer
    return lid

In [8]:
from jupytergis_lab.notebook.symbology import field

links = project.network.links.data
gdf = links.merge(comparison.reset_index(), on="link_id")
gdf["growth_pct"] = gdf["growth_pct"].fillna(0)

doc = new_map(gdf, zoom=12)
lim = float(np.nanmax(np.abs(gdf["growth_pct"])))
add_gdf(doc, gdf[["link_id", "growth_pct", "base", "future", "geometry"]], "flow growth %",
        symbology=[[field("growth_pct").colormap("RdBu_r", domain=(-lim, lim)).encoding("stroke"),
                    field("future").scalar(domain=(0.0, float(gdf["future"].max())),
                                           output_range=(0.8, 7.0)).encoding("stroke-width")]])
doc

C:\Users\Riz\AppData\Local\Temp\ipykernel_29004\1581409630.py:24: UserWarning: The JupyterGIS Python API is better experienced in the xeus-python kernel which supports awaiting comm messages
  doc = GISDocument(**kwargs)


In [9]:
project.close()

---
That completes the series. Recap of the toolkit:

| Notebook | Modeling stage | Key classes |
|---|---|---|
| 01 | Project & network | `Project`, `Network` |
| 02 | Zoning & connectors | `Zoning`, `Zone.connect_mode` |
| 03 | Paths & skims | `Graph`, `PathResults`, `NetworkSkimming` |
| 04 | Distribution | `GravityCalibration`, `Ipf`, `GravityApplication` |
| 05 | Assignment | `TrafficAssignment`, `TrafficClass` |
| 06 | Route choice | `RouteChoice` |
| 07 | Transit | `Transit`, GTFS builder |
| 08 | Forecasting | everything together |

Further reading: the [AequilibraE documentation](https://www.aequilibrae.com) —
every notebook here mirrors one or more of its worked examples.
